# Composite Plate Analysis - Complete Surrogate Modeling
## AEROSP 740 Project - Vijai Venkatesh Natarajan Ganesh Babu

**Analysis Mode:** Set `Config.ANALYSIS_MODE` in the Configuration cell:
- `"a"` — Buckling Load Factor
- `"b"` — Bending Deflection (center-point, uniform pressure)
- `"c"` — Natural Frequency (first mode)

**Key Features:**
- Physics-based model using ComposiPy
- Morris Screening for variable importance
- Latin Hypercube Sampling for design space exploration
- Polynomial Response Surface Model (RSM)
- Comprehensive overfitting diagnostics
- Sobol global sensitivity analysis

## 1. Import Libraries

In [ ]:
# Standard Libraries
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Sampling and DoE
from scipy.stats import qmc
from SALib.sample import morris as morris_sample
from SALib.analyze import morris as morris_analyze
from SALib.analyze import sobol

# ComposiPy for composite analysis
from composipy import OrthotropicMaterial, LaminateProperty, PlateStructure
from scipy.sparse.linalg import ArpackError

# Bending/frequency analysis
from scipy.linalg import eigh
from scipy.integrate import quad
from composipy.pre_integrated_component.functions import ii_ff, sxieta, fxi

# Machine Learning
from sklearn.preprocessing import PolynomialFeatures, MinMaxScaler
from sklearn.model_selection import train_test_split, learning_curve, cross_val_score
from sklearn.linear_model import RidgeCV, Ridge, LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Statistics
from scipy.stats import shapiro

# Set random seed for reproducibility
np.random.seed(42)

# Configure plotting
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 10

print("All libraries imported successfully")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

## 2. Configuration

In [ ]:
class Config:
    """Central configuration for composite plate analysis"""
    
    # ===== ANALYSIS MODE =====
    # Change this ONE variable to switch the entire analysis:
    #   "a" = Buckling Load Factor
    #   "b" = Bending Deflection (center-point, uniform pressure)
    #   "c" = Natural Frequency (first mode of free vibration)
    ANALYSIS_MODE = "a"
    
    MODE_META = {
        "a": {"name": "Buckling Load Factor", "short": "BLF", "unit": ""},
        "b": {"name": "Bending Deflection", "short": "BD", "unit": "m"},
        "c": {"name": "Natural Frequency", "short": "NF", "unit": "Hz"},
    }
    
    RESPONSE_NAME = MODE_META[ANALYSIS_MODE]["name"]
    RESPONSE_SHORT = MODE_META[ANALYSIS_MODE]["short"]
    RESPONSE_UNIT = MODE_META[ANALYSIS_MODE]["unit"]
    
    # Material Properties
    POISSON_RATIO = 0.33
    STACKING_LAYERS = 8
    
    # Bending mode: uniform transverse pressure (Pa)
    PRESSURE_LOAD = 1000.0
    
    # Design Space - 15 variables
    DESIGN_SPACE = {
        "Plate Length (mm)": [200, 500],
        "Plate Width (mm)": [200, 500],
        "Thickness (mm)": [0.1, 1.5],
        "Young's Modulus (Fiber) (MPa)": [40000, 80000],
        "Young's Modulus (Transverse) (MPa)": [40000, 80000],
        "Material Density (kg/m3)": [1500, 2000],
        "Layer 1 Angle (deg)": [0, 180],
        "Layer 2 Angle (deg)": [0, 180],
        "Layer 3 Angle (deg)": [0, 180],
        "Layer 4 Angle (deg)": [0, 180],
        "Layer 5 Angle (deg)": [0, 180],
        "Layer 6 Angle (deg)": [0, 180],
        "Layer 7 Angle (deg)": [0, 180],
        "Layer 8 Angle (deg)": [0, 180],
        "Boundary Condition": [0, 3]
    }
    
    BC_MAP = {0: "Pinned", 1: "Clamped", 2: "Free Edge", 3: "Shear"}
    
    COMPOSIPY_BC = {
        "Pinned": {"x0": "PINNED", "xa": "PINNED", "y0": "PINNED", "yb": "PINNED"},
        "Clamped": {"x0": "CLAMPED", "xa": "CLAMPED", "y0": "CLAMPED", "yb": "PINNED"},
        "Free Edge": {"x0": "FREE", "xa": "FREE", "y0": "FREE", "yb": "FREE"},
        "Shear": {"x0": "SIMPLY_SUPPORTED", "xa": "SIMPLY_SUPPORTED",
                  "y0": "SIMPLY_SUPPORTED", "yb": "SIMPLY_SUPPORTED"}
    }
    
    LAYER_ANGLE_COLS = [f"Layer {i} Angle (deg)" for i in range(1, STACKING_LAYERS + 1)]
    
    CONTINUOUS_COLS = [
        "Plate Length (mm)", "Plate Width (mm)", "Thickness (mm)",
        "Young's Modulus (Fiber) (MPa)", "Young's Modulus (Transverse) (MPa)",
        "Material Density (kg/m3)",
    ] + LAYER_ANGLE_COLS
    
    LHS_SAMPLES = 3500
    POLYNOMIAL_DEGREE = 2
    TEST_SIZE = 0.15
    VALIDATION_SIZE = 0.15
    RANDOM_STATE = 42
    
    PLATE_M = 10
    PLATE_N = 10
    NXX_LOAD = -1.0
    
    OUTPUT_DIR = Path("buckling_analysis_outputs")

# Module-level convenience aliases
RNAME = Config.RESPONSE_NAME
RSHORT = Config.RESPONSE_SHORT
RUNIT = Config.RESPONSE_UNIT

Config.OUTPUT_DIR.mkdir(exist_ok=True)
print(f"Configuration loaded - Mode: {Config.ANALYSIS_MODE} ({RNAME})")
print(f"  Output directory: {Config.OUTPUT_DIR}")
print(f"  Design variables: {len(Config.DESIGN_SPACE)} (including 8 independent layer angles)")
print(f"  Layer angles: continuous [0, 180] degrees (asymmetric laminates supported)")
print(f"  Polynomial degree: {Config.POLYNOMIAL_DEGREE}")
print(f"  LHS samples: {Config.LHS_SAMPLES}")


## 3. Utility Functions

In [ ]:
def normalize_data(df, columns):
    """Min-Max normalize specified columns"""
    df_scaled = df.copy()
    for col in columns:
        col_min, col_max = df[col].min(), df[col].max()
        if col_max - col_min > 0:
            df_scaled[col] = (df[col] - col_min) / (col_max - col_min)
    return df_scaled

def map_categorical(df):
    """Map categorical variables to string labels (only Boundary Condition)"""
    df_mapped = df.copy()
    if "Boundary Condition" in df_mapped.columns:
        df_mapped["Boundary Condition"] = df_mapped["Boundary Condition"].round().astype(int).map(Config.BC_MAP)
    return df_mapped

print("Utility functions defined")

## 4. Physics-Based Model

In [ ]:
def run_physics_analysis(design_params):
    """
    Run physics-based analysis using ComposiPy.
    Dispatches to buckling, bending, or frequency based on Config.ANALYSIS_MODE.
    """
    a = float(design_params["Plate Length (mm)"]) / 1000
    b = float(design_params["Plate Width (mm)"]) / 1000
    t_ply = float(design_params["Thickness (mm)"]) / 1000
    E1 = float(design_params["Young's Modulus (Fiber) (MPa)"]) * 1e6
    E2 = float(design_params["Young's Modulus (Transverse) (MPa)"]) * 1e6
    rho = float(design_params["Material Density (kg/m3)"])
    bc_str = str(design_params["Boundary Condition"])
    
    v12 = Config.POISSON_RATIO
    G12 = E1 / (2 * (1 + v12))
    h_total = t_ply * Config.STACKING_LAYERS
    
    stacking_sequence = [
        float(design_params[f"Layer {i} Angle (deg)"])
        for i in range(1, Config.STACKING_LAYERS + 1)
    ]
    
    material = OrthotropicMaterial(E1, E2, v12, G12, t_ply)
    laminate = LaminateProperty(stacking_sequence, material)
    
    if bc_str not in Config.COMPOSIPY_BC:
        raise ValueError(f"Unknown BC: {bc_str}")
    constraints = Config.COMPOSIPY_BC[bc_str]
    
    plate = PlateStructure(
        laminate, a, b,
        m=Config.PLATE_M, n=Config.PLATE_N,
        Nxx=Config.NXX_LOAD,
        constraints=constraints
    )
    
    if Config.ANALYSIS_MODE == "a":
        return _buckling_analysis(plate)
    elif Config.ANALYSIS_MODE == "b":
        return _bending_analysis(plate, a, b)
    elif Config.ANALYSIS_MODE == "c":
        return _frequency_analysis(plate, a, b, rho, h_total)
    else:
        raise ValueError(f"Unknown ANALYSIS_MODE: {Config.ANALYSIS_MODE}")


def _buckling_analysis(plate):
    """Buckling Load Factor: first eigenvalue from linear buckling analysis."""
    eigenvals, _ = plate.buckling_analysis()
    return eigenvals[0]


def _bending_analysis(plate, a, b):
    """Bending Deflection: center-point deflection under uniform pressure."""
    K, _ = plate.calc_K_KG_D()
    sw_idx = plate.sw_idx
    n_dof = len(sw_idx)
    p = Config.PRESSURE_LOAD
    
    # Cache shape function integrals: integral of f_i(xi) over [-1, 1]
    unique_indices = set()
    for (i, j) in sw_idx:
        unique_indices.add(i)
        unique_indices.add(j)
    
    sf_integrals = {}
    for idx in unique_indices:
        val, _ = quad(lambda xi, _i=idx: fxi(_i, xi), -1, 1)
        sf_integrals[idx] = val
    
    # Load vector: F_r = p * (a*b/4) * integral_i * integral_j
    F = np.zeros(n_dof)
    for r, (i, j) in enumerate(sw_idx):
        F[r] = p * (a * b / 4.0) * sf_integrals[i] * sf_integrals[j]
    
    w_coeffs = np.linalg.solve(K, F)
    
    # Evaluate center-point deflection
    sw_center = sxieta(sw_idx, 0.0, 0.0)
    center_deflection = float(sw_center @ w_coeffs)
    
    return abs(center_deflection)


def _frequency_analysis(plate, a, b, rho, h_total):
    """Natural Frequency: first natural frequency of free vibration."""
    K, _ = plate.calc_K_KG_D()
    sw_idx = plate.sw_idx
    n_dof = len(sw_idx)
    
    # Build consistent mass matrix: M_rc = rho * h * (a*b/4) * ii_ff(i1,i2) * ii_ff(j1,j2)
    M = np.zeros((n_dof, n_dof))
    mass_factor = rho * h_total * (a * b / 4.0)
    
    for r, (i1, j1) in enumerate(sw_idx):
        for c, (i2, j2) in enumerate(sw_idx):
            M[r, c] = mass_factor * ii_ff((i1, i2)) * ii_ff((j1, j2))
    
    # Solve K * x = omega^2 * M * x
    eigenvalues, _ = eigh(K, M)
    
    positive_eigs = eigenvalues[eigenvalues > 0]
    if len(positive_eigs) == 0:
        raise ValueError("No positive eigenvalues found")
    
    omega_sq = positive_eigs[0]
    freq = np.sqrt(omega_sq) / (2 * np.pi)
    
    return freq


print(f"Physics model defined - Mode: {Config.ANALYSIS_MODE} ({RNAME})")


## 5. Morris Screening (Elementary Effects)

**Purpose:** Efficient one-at-a-time (OAT) screening method to identify which design variables have the most influence on the selected response metric, before running the full LHS campaign.

**Key Metrics:**
- **mu_star** (|mu*|): Absolute mean of elementary effects — measures overall importance
- **sigma**: Standard deviation of elementary effects — measures nonlinearity / interaction effects

**Important Note on Layer Angles:** Morris OAT perturbs one variable at a time. Since the stacking sequence effect is distributed across 8 layer variables, individual layer mu* values appear small. A **grouped analysis** sums the 8 layer contributions to reveal the true combined influence of fiber orientation.

In [ ]:
print(f"\n{'='*70}")
print(f"MORRIS SCREENING (ELEMENTARY EFFECTS)")
print(f"{'='*70}")

# Define problem in actual design space bounds
problem_morris = {
    'num_vars': len(Config.DESIGN_SPACE),
    'names': list(Config.DESIGN_SPACE.keys()),
    'bounds': list(Config.DESIGN_SPACE.values())
}

# Generate Morris trajectories
N_morris = 60  # Number of trajectories
X_morris = morris_sample.sample(problem_morris, N=N_morris, num_levels=4)

total_morris = len(X_morris)
print(f"Generated {total_morris} Morris samples ({N_morris} trajectories, {problem_morris['num_vars']} variables)")
print(f"Running physics model on Morris samples...\n")

# Evaluate physics model on each Morris sample
y_morris = np.zeros(total_morris)
morris_failed = 0
checkpoint_m = max(1, total_morris // 5)

for i in range(total_morris):
    if (i + 1) % checkpoint_m == 0:
        print(f"  Progress: {i + 1}/{total_morris} ({(i+1)/total_morris*100:.0f}%)")

    params = dict(zip(problem_morris['names'], X_morris[i]))
    params_df = pd.DataFrame([params])
    params_df = map_categorical(params_df)
    params_dict = params_df.iloc[0].to_dict()

    try:
        result = run_physics_analysis(params_dict)
        # Use log transform; treat non-positive values as failure
        y_morris[i] = np.log(max(result, 1e-30))
    except Exception:
        y_morris[i] = np.log(1e-30)  # log floor for failed plates
        morris_failed += 1

print(f"\nMorris evaluation complete: {total_morris - morris_failed} succeeded, {morris_failed} failed")

# Run Morris analysis
Si_morris = morris_analyze.analyze(
    problem_morris, X_morris, y_morris,
    num_resamples=1000,
    conf_level=0.95,
    print_to_console=False
)

# Print results table
print(f"\n{'='*70}")
print(f"{'Parameter':<35} {'mu*':<12} {'sigma':<12} {'mu*_conf':<12}")
print(f"{'-'*70}")

# Sort by mu_star descending
sort_idx = np.argsort(Si_morris['mu_star'])[::-1]
for idx in sort_idx:
    name = problem_morris['names'][idx]
    print(f"{name:<35} {Si_morris['mu_star'][idx]:<12.4f} {Si_morris['sigma'][idx]:<12.4f} {Si_morris['mu_star_conf'][idx]:<12.4f}")
print(f"{'='*70}")

# Classify variables
threshold = 0.1 * Si_morris['mu_star'].max()  # 10% of max mu*
important_vars = [problem_morris['names'][i] for i in range(problem_morris['num_vars'])
                  if Si_morris['mu_star'][i] >= threshold]
negligible_vars = [problem_morris['names'][i] for i in range(problem_morris['num_vars'])
                   if Si_morris['mu_star'][i] < threshold]

print(f"\nVariable Classification (threshold = 10% of max mu*):")
print(f"  Important ({len(important_vars)}): {', '.join(important_vars)}")
if negligible_vars:
    print(f"  Negligible ({len(negligible_vars)}): {', '.join(negligible_vars)}")
else:
    print(f"  All variables are influential")

# === VISUALIZATION ===
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Plot 1: mu* vs sigma scatter (Morris plot)
ax1 = axes[0]
ax1.scatter(Si_morris['mu_star'], Si_morris['sigma'], s=80, c='steelblue',
            edgecolors='navy', linewidth=0.8, zorder=5)

# Label each point
for i, name in enumerate(problem_morris['names']):
    short = name.replace(' (mm)', '').replace(' (MPa)', '').replace(' (kg/m3)', '')
    short = short.replace("Young's Modulus ", 'E_').replace('(Fiber)', 'f').replace('(Transverse)', 't')
    short = short.replace('Plate ', '').replace('Material ', '').replace(' (deg)', '')
    ax1.annotate(short, (Si_morris['mu_star'][i], Si_morris['sigma'][i]),
                 textcoords="offset points", xytext=(5, 5), fontsize=7, alpha=0.85)

# Add sigma = mu* line (indicates nonlinearity)
max_val = max(Si_morris['mu_star'].max(), Si_morris['sigma'].max()) * 1.1
ax1.plot([0, max_val], [0, max_val], 'r--', alpha=0.4, label='sigma = mu* (nonlinear)')

# Add threshold line
ax1.axvline(x=threshold, color='orange', linestyle=':', alpha=0.6, label=f'10% threshold = {threshold:.2f}')

ax1.set_xlabel('mu* (Absolute Mean of Elementary Effects)', fontsize=11)
ax1.set_ylabel('sigma (Std Dev of Elementary Effects)', fontsize=11)
ax1.set_title('Morris Screening: mu* vs sigma\n(upper-right = important & nonlinear)', fontsize=12, fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# Plot 2: Horizontal bar chart of mu* ranked
ax2 = axes[1]
sorted_names = [problem_morris['names'][i] for i in sort_idx]
sorted_mu_star = Si_morris['mu_star'][sort_idx]
sorted_conf = Si_morris['mu_star_conf'][sort_idx]

short_names = []
for name in sorted_names:
    short = name.replace(' (mm)', '').replace(' (MPa)', '').replace(' (kg/m3)', '')
    short = short.replace("Young's Modulus ", 'E_').replace('(Fiber)', 'fiber').replace('(Transverse)', 'trans')
    short = short.replace('Material ', '').replace(' (deg)', '')
    short_names.append(short)

colors = ['steelblue' if mu >= threshold else 'lightgray' for mu in sorted_mu_star]
bars = ax2.barh(range(len(sorted_names)), sorted_mu_star, color=colors, edgecolor='navy', linewidth=0.5)
ax2.errorbar(sorted_mu_star, range(len(sorted_names)), xerr=sorted_conf,
             fmt='none', color='black', capsize=3, linewidth=0.8)

ax2.set_yticks(range(len(short_names)))
ax2.set_yticklabels(short_names, fontsize=9)
ax2.set_xlabel('mu* (Absolute Mean of Elementary Effects)', fontsize=11)
ax2.set_title('Morris Screening: Variable Ranking\n(gray = below 10% threshold)', fontsize=12, fontweight='bold')
ax2.axvline(x=threshold, color='orange', linestyle=':', alpha=0.6)
ax2.invert_yaxis()
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(Config.OUTPUT_DIR / 'morris_screening.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nMorris screening complete - saved to {Config.OUTPUT_DIR / 'morris_screening.png'}")

# === GROUPED ANALYSIS: Combined Stacking Sequence Effect ===
print(f"\n{'='*70}")
print(f"GROUPED VARIABLE ANALYSIS")
print(f"{'='*70}")
print(f"\nNote: Morris OAT screening perturbs ONE variable at a time.")
print(f"Individual layer angles show low mu* because changing 1 of 8 plies")
print(f"has a modest effect. The COMBINED stacking sequence effect is much larger.\n")

# Group layer angles vs other variables
angle_indices = [i for i, n in enumerate(problem_morris['names']) if 'Layer' in n and 'Angle' in n]
non_angle_names = []
non_angle_mu_star = []
non_angle_sigma = []

for i, name in enumerate(problem_morris['names']):
    if i not in angle_indices:
        non_angle_names.append(name)
        non_angle_mu_star.append(Si_morris['mu_star'][i])
        non_angle_sigma.append(Si_morris['sigma'][i])

# Combined stacking sequence metrics (sum captures total orientation influence)
combined_mu_star = np.sum(Si_morris['mu_star'][angle_indices])
combined_sigma = np.sqrt(np.sum(Si_morris['sigma'][angle_indices]**2))  # RMS for sigma
mean_mu_star = np.mean(Si_morris['mu_star'][angle_indices])

# Build grouped results
group_names = non_angle_names + ['Stacking Sequence\n(8 angles combined)']
group_mu_star = non_angle_mu_star + [combined_mu_star]
group_sigma = non_angle_sigma + [combined_sigma]

# Print grouped table
print(f"{'Variable Group':<35} {'mu*':<12} {'sigma':<12}")
print(f"{'-'*60}")
for name, mu, sig in zip(group_names, group_mu_star, group_sigma):
    clean_name = name.replace('\n', ' ')
    print(f"{clean_name:<35} {mu:<12.4f} {sig:<12.4f}")
print(f"{'='*60}")

# Grouped bar chart
fig, ax = plt.subplots(figsize=(12, 6))

short_group_names = []
for name in group_names:
    short = name.replace(' (mm)', '').replace(' (MPa)', '').replace(' (kg/m3)', '')
    short = short.replace("Young's Modulus ", 'E_').replace('(Fiber)', 'fiber').replace('(Transverse)', 'trans')
    short = short.replace('Material ', '').replace(' (deg)', '')
    short_group_names.append(short)

# Color: stacking sequence in orange, others in steelblue
colors_g = ['steelblue'] * len(non_angle_names) + ['darkorange']
sorted_g = np.argsort(group_mu_star)[::-1]

bars = ax.barh(range(len(group_names)),
               [group_mu_star[i] for i in sorted_g],
               color=[colors_g[i] for i in sorted_g],
               edgecolor='navy', linewidth=0.5)

ax.set_yticks(range(len(short_group_names)))
ax.set_yticklabels([short_group_names[i] for i in sorted_g], fontsize=10)
ax.set_xlabel('mu* (Absolute Mean of Elementary Effects)', fontsize=11)
ax.set_title('Morris Screening: Grouped Variable Importance\n(Orange = combined effect of all 8 layer angles)',
             fontsize=12, fontweight='bold')
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)

# Add value labels on bars
for bar, val in zip(bars, [group_mu_star[i] for i in sorted_g]):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}', va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig(Config.OUTPUT_DIR / 'morris_grouped.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nKey Insight: When the 8 layer angles are grouped as 'Stacking Sequence',")
print(f"their combined mu* = {combined_mu_star:.2f} makes fiber orientation")
print(f"one of the most influential factor groups for {RNAME}.")
print(f"This aligns with composite mechanics: {RNAME} depends on the FULL stacking sequence,")
print(f"not individual ply angles in isolation.")

## 6. Generate LHS Samples

In [ ]:
print(f"Generating {Config.LHS_SAMPLES} LHS samples...")

# Initialize LHS sampler
sampler = qmc.LatinHypercube(d=len(Config.DESIGN_SPACE), seed=Config.RANDOM_STATE)
lhs_samples = sampler.random(n=Config.LHS_SAMPLES)

# Scale to design space bounds
bounds = np.array(list(Config.DESIGN_SPACE.values()))
lower_bounds = bounds[:, 0]
upper_bounds = bounds[:, 1]
scaled_samples = qmc.scale(lhs_samples, lower_bounds, upper_bounds)

# Create DataFrame
lhs_df = pd.DataFrame(scaled_samples, columns=list(Config.DESIGN_SPACE.keys()))
lhs_df = map_categorical(lhs_df)

print(f"✓ Generated {len(lhs_df)} design points")
print(f"\nFirst 5 samples:")
print(lhs_df.head())

## 7. Execute Physics Model

In [ ]:
print(f"\nRunning {RNAME} analysis on {len(lhs_df)} samples...")
print("This may take several minutes...\n")

analysis_results = []
successful_indices = []
failed_count = 0

# Progress tracking
total = len(lhs_df)
checkpoint = max(1, total // 10)

for idx, row in lhs_df.iterrows():
    if (idx + 1) % checkpoint == 0:
        print(f"  Progress: {idx + 1}/{total} ({(idx+1)/total*100:.1f}%)")
    
    try:
        result = run_physics_analysis(row.to_dict())
        analysis_results.append(result)
        successful_indices.append(idx)
    except ArpackError:
        failed_count += 1
        if failed_count <= 5:
            print(f"  ⚠ Index {idx}: ARPACK convergence failure")
    except Exception as e:
        failed_count += 1
        if failed_count <= 5:
            print(f"  ✗ Index {idx}: {str(e)[:60]}")

# Create results arrays
X = lhs_df.iloc[successful_indices].reset_index(drop=True)
y = np.array(analysis_results)

# Summary
success_rate = len(successful_indices) / total * 100
print(f"\n{'='*70}")
print(f"SIMULATION COMPLETE")
print(f"{'='*70}")
print(f"  Successful: {len(successful_indices)} ({success_rate:.1f}%)")
print(f"  Failed:     {failed_count} ({100-success_rate:.1f}%)")
print(f"\n{RNAME} Statistics:")
print(f"  Mean:  {y.mean():.4f}")
print(f"  Std:   {y.std():.4f}")
print(f"  Min:   {y.min():.4f}")
print(f"  Max:   {y.max():.4f}")
print(f"  Range: {y.max() - y.min():.4f}")
print(f"{'='*70}")

## 8. Prepare Data for Modeling

In [ ]:
# Filter out invalid response values (non-positive values are physics failures)
valid_mask = y > 0
X_valid = X[valid_mask].reset_index(drop=True)
y_valid = y[valid_mask]

print(f"Filtering invalid {RSHORT} values ({RSHORT} <= 0):")
print(f"  Before: {len(X)} samples")
print(f"  Removed: {len(X) - len(X_valid)} samples with {RSHORT} <= 0")
print(f"  After:  {len(X_valid)} samples")

# Apply log-transform to reduce skewness (response ranges over several orders of magnitude)
y_log = np.log(y_valid)

print(f"\nLog-transform applied (y_log = log({RSHORT})):")
print(f"  Original range: [{y_valid.min():.2f}, {y_valid.max():.2f}]")
print(f"  Log range:      [{y_log.min():.4f}, {y_log.max():.4f}]")

# Encode categorical variables (only Boundary Condition needs reverse mapping)
X_encoded = X_valid.copy()

bc_reverse_map = {v: k for k, v in Config.BC_MAP.items()}
X_encoded["Boundary Condition"] = X_encoded["Boundary Condition"].map(bc_reverse_map)

# Normalize continuous variables (includes layer angles now)
X_normalized = normalize_data(X_encoded, Config.CONTINUOUS_COLS)

print(f"\nData prepared for modeling")
print(f"  Input features: {X_normalized.shape[1]}")
print(f"  Valid samples:  {len(X_normalized)}")
print(f"  Continuous cols normalized: {len(Config.CONTINUOUS_COLS)} (incl. 8 layer angles)")

## 9. Train Response Surface Model

In [ ]:
print(f"\n{'='*70}")
print(f"TRAINING RESPONSE SURFACE MODEL")
print(f"{'='*70}")
print(f"Target: log({RNAME})")
print(f"Polynomial degree: {Config.POLYNOMIAL_DEGREE}")

# Split data using log-transformed target
X_train, X_temp, y_train, y_temp = train_test_split(
    X_normalized, y_log,
    test_size=Config.TEST_SIZE + Config.VALIDATION_SIZE,
    random_state=Config.RANDOM_STATE
)

val_ratio = Config.VALIDATION_SIZE / (Config.TEST_SIZE + Config.VALIDATION_SIZE)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=val_ratio,
    random_state=Config.RANDOM_STATE
)

print(f"\nData Split:")
print(f"  Training:   {len(X_train)} samples ({len(X_train)/len(X_normalized)*100:.1f}%)")
print(f"  Validation: {len(X_val)} samples ({len(X_val)/len(X_normalized)*100:.1f}%)")
print(f"  Test:       {len(X_test)} samples ({len(X_test)/len(X_normalized)*100:.1f}%)")

# Generate polynomial features
poly = PolynomialFeatures(degree=Config.POLYNOMIAL_DEGREE, include_bias=True)
X_train_poly = poly.fit_transform(X_train)
X_val_poly = poly.transform(X_val)
X_test_poly = poly.transform(X_test)

print(f"\nPolynomial Features: {X_train_poly.shape[1]}")
print(f"Samples per feature: {len(X_train) / X_train_poly.shape[1]:.1f}")

# Train Ridge regression with CV
alphas = np.logspace(-6, 6, 50)
ridge_cv = RidgeCV(alphas=alphas, cv=5, scoring='neg_mean_squared_error')
ridge_cv.fit(X_train_poly, y_train)

print(f"\nOptimal Ridge alpha: {ridge_cv.alpha_:.4e}")

# Predictions (in log space)
y_train_pred = ridge_cv.predict(X_train_poly)
y_val_pred = ridge_cv.predict(X_val_poly)
y_test_pred = ridge_cv.predict(X_test_poly)

# Metrics in log space
def print_metrics(y_true, y_pred, name):
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = np.mean(np.abs(y_true - y_pred))
    print(f"\n{name} (log space):")
    print(f"  R2:   {r2:.6f}")
    print(f"  RMSE: {rmse:.6f}")
    print(f"  MAE:  {mae:.6f}")

print_metrics(y_train, y_train_pred, "Training Metrics")
print_metrics(y_val, y_val_pred, "Validation Metrics")
print_metrics(y_test, y_test_pred, "Test Metrics")

# Also show metrics in original BLF space
y_test_orig = np.exp(y_test)
y_test_pred_orig = np.exp(y_test_pred)
r2_orig = r2_score(y_test_orig, y_test_pred_orig)
rmse_orig = np.sqrt(mean_squared_error(y_test_orig, y_test_pred_orig))
print(f"\nTest Metrics (original {RSHORT} space):")
print(f"  R2:   {r2_orig:.6f}")
print(f"  RMSE: {rmse_orig:.2f}")

# Cross-validation RMSE
cv_scores = cross_val_score(
    ridge_cv, X_train_poly, y_train,
    cv=5, scoring='neg_mean_squared_error'
)
cv_rmse = np.sqrt(-cv_scores.mean())
print(f"\n5-Fold CV RMSE (log space): {cv_rmse:.6f}")
print(f"{'='*70}")

## 10. Visualize Model Performance

In [ ]:
# Actual vs Predicted (log space)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

datasets = [
    (y_train, y_train_pred, "Training"),
    (y_val, y_val_pred, "Validation"),
    (y_test, y_test_pred, "Test")
]

for ax, (y_true, y_pred, name) in zip(axes, datasets):
    ax.scatter(y_pred, y_true, alpha=0.5, s=20)
    
    # Perfect prediction line
    min_val = min(y_true.min(), y_pred.min())
    max_val = max(y_true.max(), y_pred.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect')
    
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    
    ax.set_xlabel(f'Predicted log(1+{RSHORT})', fontsize=11)
    ax.set_ylabel(f'Actual log(1+{RSHORT})', fontsize=11)
    ax.set_title(f'{name}\nR2 = {r2:.6f}, RMSE = {rmse:.6f}', fontsize=12)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle(f'RSM Performance: log(1 + {RNAME})', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(Config.OUTPUT_DIR / 'rsm_predictions.png', dpi=300, bbox_inches='tight')
plt.show()

# Residuals (log space)
residuals = y_test - y_test_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residuals vs Predicted
axes[0].scatter(y_test_pred, residuals, alpha=0.5, s=20)
axes[0].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[0].set_xlabel(f'Predicted log(1+{RSHORT})', fontsize=11)
axes[0].set_ylabel('Residuals', fontsize=11)
axes[0].set_title('Residuals vs Predicted (Test Set)', fontsize=12)
axes[0].grid(True, alpha=0.3)

# Histogram
axes[1].hist(residuals, bins=30, edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Residuals', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
axes[1].set_title('Distribution of Residuals', fontsize=12)
axes[1].axvline(x=0, color='r', linestyle='--', linewidth=2)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(Config.OUTPUT_DIR / 'residuals.png', dpi=300, bbox_inches='tight')
plt.show()

## 11. Overfitting Diagnostics

In [ ]:
print(f"\n{'='*70}")
print(f"COMPREHENSIVE OVERFITTING DIAGNOSTICS")
print(f"{'='*70}")

# 1. R² Degradation
r2_train = r2_score(y_train, y_train_pred)
r2_val = r2_score(y_val, y_val_pred)
r2_test = r2_score(y_test, y_test_pred)

r2_drop_val = r2_train - r2_val
r2_drop_test = r2_train - r2_test

print(f"\n1. R² DEGRADATION CHECK:")
print(f"   Training R²:   {r2_train:.6f}")
print(f"   Validation R²: {r2_val:.6f}")
print(f"   Test R²:       {r2_test:.6f}")
print(f"   Train → Val drop:  {r2_drop_val:.6f}")
print(f"   Train → Test drop: {r2_drop_test:.6f}")

if r2_drop_val > 0.05 or r2_drop_test > 0.05:
    print(f"   ⚠️  WARNING: Significant R² drop → OVERFITTING!")
    risk = 'HIGH'
elif abs(r2_drop_val) < 0.001 and abs(r2_drop_test) < 0.001:
    print(f"   ⚠️  WARNING: Too little drop → DATA LEAKAGE!")
    risk = 'DATA_LEAKAGE'
elif r2_train > 0.999 and r2_test > 0.999:
    print(f"   ⚠️  WARNING: R² > 0.999 → DETERMINISTIC!")
    risk = 'DETERMINISTIC'
else:
    print(f"   ✓ Normal degradation pattern")
    risk = 'LOW'

# 2. Output Variance
y_var = np.var(y_train)
y_std = np.std(y_train)
y_range = y_train.max() - y_train.min()

print(f"\n2. OUTPUT VARIANCE CHECK:")
print(f"   Variance:  {y_var:.6f}")
print(f"   Std dev:   {y_std:.6f}")
print(f"   Range:     {y_range:.6f}")
print(f"   Min:       {y_train.min():.6f}")
print(f"   Max:       {y_train.max():.6f}")

if y_var < 0.001:
    print(f"   ⚠️  WARNING: Very low variance!")
else:
    print(f"   ✓ Sufficient variation")

# 3. Relative Error
rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))
rel_rmse = (rmse_test / y_range * 100) if y_range > 0 else 0

print(f"\n3. RELATIVE ERROR CHECK:")
print(f"   Test RMSE: {rmse_test:.6f}")
print(f"   Relative:  {rel_rmse:.2f}% of range")

if rel_rmse < 0.1:
    print(f"   ⚠️  WARNING: < 0.1% → Suspiciously good!")
elif rel_rmse > 20:
    print(f"   ⚠️  WARNING: > 20% → Poor fit")
else:
    print(f"   ✓ Reasonable error level")

# 4. Model Complexity
n_samples = len(X_train)
n_features = X_train_poly.shape[1]
samples_per_feature = n_samples / n_features

print(f"\n4. MODEL COMPLEXITY CHECK:")
print(f"   Training samples:     {n_samples}")
print(f"   Polynomial features:  {n_features}")
print(f"   Samples per feature:  {samples_per_feature:.1f}")

if samples_per_feature < 5:
    print(f"   ⚠️  WARNING: Too many features!")
elif samples_per_feature < 10:
    print(f"   ⚠️  CAUTION: Borderline ratio")
else:
    print(f"   ✓ Adequate samples per feature")

# 5. Residual Normality
_, p_value = shapiro(residuals[:5000] if len(residuals) > 5000 else residuals)

print(f"\n5. RESIDUAL NORMALITY TEST:")
print(f"   Shapiro-Wilk p-value: {p_value:.4f}")
print(f"   Mean residual: {residuals.mean():.6f}")
print(f"   Std residual:  {residuals.std():.6f}")

if p_value < 0.05:
    print(f"   ⚠️  Residuals not normal (p < 0.05)")
else:
    print(f"   ✓ Residuals approximately normal")

# Overall Assessment
print(f"\n{'='*70}")
print(f"OVERALL RISK ASSESSMENT: {risk}")
print(f"{'='*70}")

## 12. Learning Curves

In [ ]:
print("\nGenerating learning curves...")

train_sizes = np.linspace(0.1, 1.0, 10)
train_sizes_abs, train_scores, val_scores = learning_curve(
    ridge_cv, X_train_poly, y_train,
    cv=5,
    scoring='neg_mean_squared_error',
    train_sizes=train_sizes,
    n_jobs=-1
)

train_rmse = np.sqrt(-train_scores)
val_rmse = np.sqrt(-val_scores)

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(train_sizes_abs, train_rmse.mean(axis=1), 
        'o-', label='Training RMSE', linewidth=2, markersize=8)
ax.fill_between(train_sizes_abs, 
                 train_rmse.mean(axis=1) - train_rmse.std(axis=1),
                 train_rmse.mean(axis=1) + train_rmse.std(axis=1),
                 alpha=0.2)

ax.plot(train_sizes_abs, val_rmse.mean(axis=1), 
        's-', label='Validation RMSE', linewidth=2, markersize=8)
ax.fill_between(train_sizes_abs, 
                 val_rmse.mean(axis=1) - val_rmse.std(axis=1),
                 val_rmse.mean(axis=1) + val_rmse.std(axis=1),
                 alpha=0.2)

ax.set_xlabel('Training Set Size', fontsize=12)
ax.set_ylabel('RMSE', fontsize=12)
ax.set_title(f'Learning Curve: {RNAME}', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

gap = val_rmse.mean(axis=1)[-1] - train_rmse.mean(axis=1)[-1]
gap_pct = gap / val_rmse.mean(axis=1)[-1] * 100

status = 'OVERFITTING!' if gap_pct > 10 else 'Good Fit'
color = 'red' if gap_pct > 10 else 'green'

ax.text(0.05, 0.95, 
        f'Final Gap: {gap:.6f} ({gap_pct:.1f}%)\n{status}',
        transform=ax.transAxes, 
        verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor=color, alpha=0.3),
        fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(Config.OUTPUT_DIR / 'learning_curve.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Learning curve generated")
print(f"  Final training RMSE:   {train_rmse.mean(axis=1)[-1]:.6f}")
print(f"  Final validation RMSE: {val_rmse.mean(axis=1)[-1]:.6f}")
print(f"  Gap: {gap:.6f} ({gap_pct:.1f}%)")

## 13. Compare Polynomial Degrees

In [ ]:
print("\nComparing polynomial degrees (1-5) on log-transformed target...\n")

degree_results = []

for degree in range(1, 6):
    # Create features
    poly_temp = PolynomialFeatures(degree=degree)
    X_train_temp = poly_temp.fit_transform(X_train)
    X_test_temp = poly_temp.transform(X_test)
    
    # Train model
    model_temp = Ridge(alpha=1.0)
    model_temp.fit(X_train_temp, y_train)
    
    # Predictions
    y_train_pred_temp = model_temp.predict(X_train_temp)
    y_test_pred_temp = model_temp.predict(X_test_temp)
    
    # Metrics (log space)
    r2_train_temp = r2_score(y_train, y_train_pred_temp)
    r2_test_temp = r2_score(y_test, y_test_pred_temp)
    rmse_test_temp = np.sqrt(mean_squared_error(y_test, y_test_pred_temp))
    
    degree_results.append({
        'Degree': degree,
        'Features': X_train_temp.shape[1],
        'Train R2': r2_train_temp,
        'Test R2': r2_test_temp,
        'Test RMSE': rmse_test_temp,
        'R2 Gap': r2_train_temp - r2_test_temp
    })

df_degrees = pd.DataFrame(degree_results)

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# R2 plot
axes[0].plot(df_degrees['Degree'], df_degrees['Train R2'], 
             'o-', label='Train R2', linewidth=2, markersize=8)
axes[0].plot(df_degrees['Degree'], df_degrees['Test R2'], 
             's-', label='Test R2', linewidth=2, markersize=8)
axes[0].set_xlabel('Polynomial Degree', fontsize=11)
axes[0].set_ylabel('R2', fontsize=11)
axes[0].set_title('R2 vs Polynomial Degree (log space)', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_xticks(df_degrees['Degree'])

# RMSE plot
axes[1].plot(df_degrees['Degree'], df_degrees['Test RMSE'], 
             'o-', linewidth=2, markersize=8, color='green')
axes[1].set_xlabel('Polynomial Degree', fontsize=11)
axes[1].set_ylabel('Test RMSE (log space)', fontsize=11)
axes[1].set_title('Test RMSE vs Polynomial Degree', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].set_xticks(df_degrees['Degree'])

# Mark optimal
optimal_idx = df_degrees['Test RMSE'].idxmin()
axes[1].plot(df_degrees.loc[optimal_idx, 'Degree'], 
             df_degrees.loc[optimal_idx, 'Test RMSE'], 
             'r*', markersize=20, 
             label=f'Optimal (degree {df_degrees.loc[optimal_idx, "Degree"]:.0f})')
axes[1].legend()

plt.tight_layout()
plt.savefig(Config.OUTPUT_DIR / 'degree_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("Polynomial Degree Comparison (log-transformed target):")
print("="*70)
print(df_degrees.to_string(index=False))
print("="*70)
print(f"\nOptimal degree: {df_degrees.loc[optimal_idx, 'Degree']:.0f} (lowest test RMSE)")
print(f"Current degree: {Config.POLYNOMIAL_DEGREE}")

if df_degrees.loc[optimal_idx, 'Degree'] != Config.POLYNOMIAL_DEGREE:
    print(f"\n  Consider changing to degree {df_degrees.loc[optimal_idx, 'Degree']:.0f}")

## 14. Baseline Comparison

In [ ]:
print("\nBaseline Model Comparison (log space):\n")

r2_test = r2_score(y_test, y_test_pred)
rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))

baseline_results = []

# 1. Current polynomial model
baseline_results.append({
    'Model': f'Polynomial (degree {Config.POLYNOMIAL_DEGREE})',
    'Features': X_train_poly.shape[1],
    'Test R2': r2_test,
    'Test RMSE': rmse_test
})

# 2. Linear regression
lr = LinearRegression()
lr.fit(X_train, y_train)
y_test_pred_lr = lr.predict(X_test)

baseline_results.append({
    'Model': 'Linear Regression',
    'Features': X_train.shape[1],
    'Test R2': r2_score(y_test, y_test_pred_lr),
    'Test RMSE': np.sqrt(mean_squared_error(y_test, y_test_pred_lr))
})

# 3. Mean baseline
y_mean = np.mean(y_train)
y_test_pred_mean = np.full_like(y_test, y_mean)

baseline_results.append({
    'Model': 'Mean Baseline',
    'Features': 0,
    'Test R2': r2_score(y_test, y_test_pred_mean),
    'Test RMSE': np.sqrt(mean_squared_error(y_test, y_test_pred_mean))
})

df_baseline = pd.DataFrame(baseline_results)

print("="*70)
print(df_baseline.to_string(index=False))
print("="*70)

if df_baseline.iloc[0]['Test R2'] - df_baseline.iloc[1]['Test R2'] < 0.05:
    print("\n  Polynomial barely better than linear!")
    print("   -> Consider using simpler linear model")
else:
    print(f"\n  Polynomial provides {(df_baseline.iloc[0]['Test R2'] - df_baseline.iloc[1]['Test R2'])*100:.1f}% R2 improvement over linear")

## 15. Sobol Sensitivity Analysis

In [ ]:
print(f"\n{'='*70}")
print(f"SOBOL GLOBAL SENSITIVITY ANALYSIS")
print(f"{'='*70}")

from SALib.sample import saltelli as saltelli_sample

# Define problem for Sobol analysis (normalized [0,1] space)
problem = {
    "num_vars": len(Config.DESIGN_SPACE),
    "names": list(Config.DESIGN_SPACE.keys()),
    "bounds": [[0.0, 1.0]] * len(Config.DESIGN_SPACE)
}

# Generate Saltelli samples (required structure for Sobol analysis)
N_sobol = 1024
X_sobol_saltelli = saltelli_sample.sample(problem, N_sobol, calc_second_order=False)

print(f"Generated {len(X_sobol_saltelli)} Saltelli samples (N={N_sobol}, D={problem['num_vars']})")

# Evaluate the trained surrogate model on Saltelli samples
X_sobol_poly = poly.transform(X_sobol_saltelli)
y_sobol = ridge_cv.predict(X_sobol_poly)

print(f"Evaluated surrogate model on {len(y_sobol)} samples")

# Perform Sobol analysis
Si = sobol.analyze(
    problem,
    y_sobol,
    calc_second_order=False,
    print_to_console=False
)

# Print results
print("\nSobol Indices:")
print("="*70)
print(f"{'Parameter':<35} {'S1 (First)':<12} {'ST (Total)':<12}")
print("-"*70)
for i, name in enumerate(problem['names']):
    print(f"{name:<35} {Si['S1'][i]:<12.4f} {Si['ST'][i]:<12.4f}")
print("="*70)

# Top 5 influential (expanded from 3 for 15 variables)
top_indices = np.argsort(Si['S1'])[::-1][:5]
print("\nTop 5 Most Influential Parameters (First-Order):")
for i, idx in enumerate(top_indices, 1):
    print(f"  {i}. {problem['names'][idx]}: S1 = {Si['S1'][idx]:.4f}, ST = {Si['ST'][idx]:.4f}")

# Plot (wider figure for 15 variables)
fig, ax = plt.subplots(figsize=(16, 7))

bar_width = 0.35
indices = np.arange(len(problem['names']))

ax.bar(indices, Si['S1'], bar_width, label='First Order (S1)', color='cornflowerblue')
ax.bar(indices + bar_width, Si['ST'], bar_width, label='Total Order (ST)', 
       alpha=0.7, color='tomato')

ax.set_xticks(indices + bar_width / 2)
ax.set_xticklabels(problem['names'], rotation=55, ha='right', fontsize=8)
ax.set_ylabel("Sobol Index", fontsize=12)
ax.set_title(f"Global Sensitivity Analysis: {RNAME}\n(Asymmetric Laminates)", 
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(Config.OUTPUT_DIR / 'sobol_indices.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nSobol analysis complete")

## 16. Final Summary & Recommendations

In [ ]:
print(f"\n{'='*70}")
print(f"FINAL SUMMARY: {RNAME.upper()} ANALYSIS")
print(f"{'='*70}")

# Recompute key metrics
r2_train = r2_score(y_train, y_train_pred)
r2_val = r2_score(y_val, y_val_pred)
r2_test = r2_score(y_test, y_test_pred)
rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))
r2_drop_test = r2_train - r2_test
n_features = X_train_poly.shape[1]
samples_per_feature = len(X_train) / n_features

y_range = y_train.max() - y_train.min()
rel_rmse = (rmse_test / y_range * 100) if y_range > 0 else 0

# Determine risk
if r2_drop_test > 0.05:
    risk = 'HIGH'
elif abs(r2_drop_test) < 0.001:
    risk = 'DATA_LEAKAGE'
elif r2_train > 0.999 and r2_test > 0.999:
    risk = 'DETERMINISTIC'
else:
    risk = 'LOW'

print(f"\n1. DATA:")
print(f"   Total LHS samples:  {Config.LHS_SAMPLES}")
print(f"   After filtering:    {len(X_normalized)} (removed {RSHORT} <= 0)")
print(f"   Training set:       {len(X_train)}")
print(f"   Validation set:     {len(X_val)}")
print(f"   Test set:           {len(X_test)}")

print(f"\n2. MODEL:")
print(f"   Type:               Response Surface Model (Ridge)")
print(f"   Target:             log({RSHORT})")
print(f"   Laminate type:      Asymmetric (8 independent layer angles)")
print(f"   Design variables:   {len(Config.DESIGN_SPACE)}")
print(f"   Polynomial degree:  {Config.POLYNOMIAL_DEGREE}")
print(f"   Features:           {n_features}")
print(f"   Samples/feature:    {samples_per_feature:.1f}")
print(f"   Optimal alpha:      {ridge_cv.alpha_:.4e}")

print(f"\n3. PERFORMANCE (log space):")
print(f"   Train R2:           {r2_train:.6f}")
print(f"   Validation R2:      {r2_val:.6f}")
print(f"   Test R2:            {r2_test:.6f}")
print(f"   Test RMSE:          {rmse_test:.6f}")
print(f"   Relative RMSE:      {rel_rmse:.2f}% of range")
print(f"   CV RMSE:            {cv_rmse:.6f}")

# Original space metrics
y_test_orig = np.exp(y_test)
y_test_pred_orig = np.exp(y_test_pred)
r2_orig = r2_score(y_test_orig, y_test_pred_orig)
rmse_orig = np.sqrt(mean_squared_error(y_test_orig, y_test_pred_orig))
print(f"\n4. PERFORMANCE (original {RSHORT} space):")
print(f"   Test R2:            {r2_orig:.6f}")
print(f"   Test RMSE:          {rmse_orig:.2f}")

print(f"\n5. OVERFITTING RISK:")
print(f"   Assessment:         {risk}")
print(f"   R2 degradation:     {r2_drop_test:.6f}")

print(f"\n6. SENSITIVITY (Top 5):")
for i, idx in enumerate(top_indices, 1):
    print(f"   {i}. {problem['names'][idx]}: S1 = {Si['S1'][idx]:.4f}")

print(f"\n7. KEY FEATURES:")
print(f"   - Asymmetric laminates: 8 independent layer angles [0, 180] degrees")
print(f"   - Continuous angle sampling (not limited to 0/45/90)")
print(f"   - Log-transform on response (reduces skewness)")
print(f"   - Filtered invalid {RSHORT} <= 0 samples")
print(f"   - Degree {Config.POLYNOMIAL_DEGREE} polynomial with Ridge regularization")
print(f"   - Sobol sensitivity via Saltelli sampling on surrogate model")

print(f"\n8. OUTPUT FILES:")
print(f"   Directory: {Config.OUTPUT_DIR}/")
print(f"   - rsm_predictions.png")
print(f"   - residuals.png")
print(f"   - learning_curve.png")
print(f"   - degree_comparison.png")
print(f"   - sobol_indices.png")
print(f"   - morris_screening.png")
print(f"   - morris_grouped.png")

print(f"\n{'='*70}")
print(f"ANALYSIS COMPLETE")
print(f"{'='*70}")